# Training across more than one GPU

Three ways to train here, and which one you want depends only on how many GPUs you need.

| GPUs | How | Where |
|---|---|---|
| 1 | Just run it in this notebook | your server |
| up to 8, one node | Spawn the **GPU — 8 x GPU** profile, then `torchrun --nproc_per_node=8` | your server |
| more than 8 | Submit a **PyTorchJob** (below) | the cluster |

The reason for the split: this notebook is *one pod*. Nothing sets `MASTER_ADDR` or
`RANK` for a second machine, and no second machine is listening. `torchrun --nnodes=4`
here will simply hang. The Training Operator creates those pods and wires them together.

A submitted job also **outlives this notebook** — close the tab, the job keeps running,
and the idle culler never touches it.


## 1. Single node, up to 8 GPUs

Nothing special needed. Put your training code in a file (not a notebook cell) and:


In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

# then, from a terminal on an 8-GPU profile:
#   torchrun --standalone --nproc_per_node=8 ~/team/train.py


## 2. More than one node: submit a PyTorchJob

Write your training script to `~/team/` — shared storage, so every worker on every
node sees the same file, and checkpoints written there come back to this notebook.

Your script needs no cluster-specific code. `torchrun` reads the environment the
operator sets:


In [ ]:
%%writefile ~/team/train.py
import os, torch, torch.distributed as dist

dist.init_process_group('nccl')            # reads MASTER_ADDR/RANK/WORLD_SIZE
rank, world = dist.get_rank(), dist.get_world_size()
torch.cuda.set_device(int(os.environ['LOCAL_RANK']))

if rank == 0:
    print(f'training on {world} GPUs', flush=True)

# ... your model, wrapped in DistributedDataParallel or FSDP ...
# Checkpoint from rank 0 only, to shared storage:
#   if rank == 0: torch.save(model.state_dict(), '/home/jovyan/team/ckpt.pt')

dist.destroy_process_group()


### Submit it

`nnodes` x `nproc_per_node` is your total GPU count. `Worker.replicas` is `nnodes - 1`
because the master counts as one.


In [ ]:
from kubernetes import client, config
import yaml, os

config.load_incluster_config()

NAME, NNODES, GPUS_PER_NODE = 'my-run', 4, 8
IMAGE = os.environ.get('JUPYTER_IMAGE_SPEC', 'pytorch-caimex:cuda12')

pod = {
  'spec': {
    'runtimeClassName': 'nvidia',
    'nodeSelector': {'caimex.io/pool': 'gpu'},   # whole GPUs, not MIG slices
    'containers': [{
      'name': 'pytorch',
      'image': IMAGE,
      'command': ['torchrun', f'--nnodes={NNODES}',
                  f'--nproc_per_node={GPUS_PER_NODE}', '/home/jovyan/team/train.py'],
      'env': [{'name': 'NCCL_DEBUG', 'value': 'WARN'}],
      'resources': {'limits': {'nvidia.com/gpu': GPUS_PER_NODE}},
      'volumeMounts': [{'name': 'team', 'mountPath': '/home/jovyan/team'},
                       {'name': 'dshm', 'mountPath': '/dev/shm'}],
    }],
    'volumes': [
      {'name': 'team', 'persistentVolumeClaim': {'claimName': 'jupyter-shared-team'}},
      {'name': 'dshm', 'emptyDir': {'medium': 'Memory', 'sizeLimit': '16Gi'}},
    ],
  }
}

job = {
  'apiVersion': 'kubeflow.org/v1', 'kind': 'PyTorchJob',
  'metadata': {'name': NAME,
               # queue-name is what gets you gang scheduling: the job starts
               # only when ALL its pods can be placed, never half of them.
               'labels': {'kueue.x-k8s.io/queue-name': 'user-queue'}},
  'spec': {'pytorchReplicaSpecs': {
     'Master': {'replicas': 1,          'restartPolicy': 'OnFailure', 'template': pod},
     'Worker': {'replicas': NNODES - 1, 'restartPolicy': 'OnFailure', 'template': pod},
  }}
}

api = client.CustomObjectsApi()
ns = open('/var/run/secrets/kubernetes.io/serviceaccount/namespace').read()
api.create_namespaced_custom_object('kubeflow.org', 'v1', ns, 'pytorchjobs', job)
print(f'submitted {NAME}: {NNODES * GPUS_PER_NODE} GPUs')


### Watch it

A job sitting in `Suspended` is not broken — it is queued, waiting for enough GPUs to
free up so that *all* of its pods can start together. `kubectl describe workload` says
what it is waiting for.


In [ ]:
!kubectl get pytorchjob,pods -l training.kubeflow.org/job-name=my-run
# !kubectl logs -f my-run-master-0
# !kubectl delete pytorchjob my-run


---
## Things that will bite you

**Ask for whole GPUs, not MIG slices.** MIG instances have no NVLink path between
them; NCCL falls back through host memory and 7 slices train slower than 1 whole GPU.
That is what `nodeSelector: caimex.io/pool=gpu` is for.

**Checkpoint from rank 0 only,** to `~/team`. All ranks writing the same file will
corrupt it, and anything written to a worker's local disk vanishes when the pod exits.

**Request what you will use.** GPUs held by a queued-but-idle job are GPUs nobody else
can have. Ask for 64 only when you have already shown the code scales at 8.

**Scaling is not free.** If 16 GPUs are not roughly twice 8, the bottleneck is almost
always the network or the data loader, not the GPUs — check before asking for more.
